# 01 Load Manual Definition

Read manual dictionary Excel from OneLake Files and create `governance.manual_*` tables.

In [ ]:
import os
import pandas as pd
from pyspark.sql import functions as F

CATALOG_SCHEMA = "governance"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_SCHEMA}")

In [ ]:
candidate_paths = [
    "/lakehouse/default/Files/data_dictionary/manual/manual_data_dictionary.xlsx",
    "/lakehouse/default/Files/data_dictionary/manual/manual_data_dictionary_filled.xlsx",
    "/lakehouse/default/Files/DataDictionary/manual/manual_data_dictionary.xlsx",
    "/lakehouse/default/Files/DataDictionary/manual/manual_data_dictionary_filled.xlsx",
]

manual_path = None
for p in candidate_paths:
    if os.path.exists(p):
        manual_path = p
        break

if manual_path is None:
    raise FileNotFoundError(
        "Manual dictionary file not found. Upload Excel file to /lakehouse/default/Files/data_dictionary/manual/"
    )

print(f"Using manual file: {manual_path}")

In [ ]:
table_pdf = pd.read_excel(manual_path, sheet_name="table_definition", dtype=str, engine="openpyxl")
column_pdf = pd.read_excel(manual_path, sheet_name="column_definition", dtype=str, engine="openpyxl")

table_pdf = table_pdf.where(pd.notnull(table_pdf), None)
column_pdf = column_pdf.where(pd.notnull(column_pdf), None)

manual_table = spark.createDataFrame(table_pdf)
manual_column = spark.createDataFrame(column_pdf)

table_key_cols = ["workspace_name", "lakehouse_name", "schema_name", "table_name"]
column_key_cols = table_key_cols + ["column_name"]

for c in table_key_cols:
    manual_table = manual_table.withColumn(c, F.trim(F.col(c)))

for c in column_key_cols:
    manual_column = manual_column.withColumn(c, F.trim(F.col(c)))

manual_column = manual_column.withColumn(
    "is_visible_in_portal",
    F.when(
        F.lower(F.trim(F.col("is_visible_in_portal").cast("string"))).isin("false", "0", "no", "n"),
        F.lit(False)
    ).otherwise(F.lit(True))
)

manual_table = manual_table.withColumn("manual_updated_at", F.current_timestamp())
manual_column = manual_column.withColumn("manual_updated_at", F.current_timestamp())

manual_table.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG_SCHEMA}.manual_table_definition")

manual_column.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG_SCHEMA}.manual_column_definition")

In [ ]:
display(spark.table(f"{CATALOG_SCHEMA}.manual_table_definition").limit(10))
display(spark.table(f"{CATALOG_SCHEMA}.manual_column_definition").limit(10))